# Draft night — Sleeper, 14-team superflex, half-PPR

Thursday 3 September 2026, 18:30. Snake, no third-round reversal, 15 rounds, 120-second pick timer,
**seat 1**. Picks: **1, 28, 29, 56, 57, 84, 85, 112, 113, 140, 141, 168, 169, 196, 197** — one pick
at the top and then seven back-to-back pairs.

Everything below is computed from the warehouse, not typed in. Re-run after a rebuild and the
findings update; if a number here moved, the draft moved.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.query import q

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

LEAGUE = "sleeper"
MY_SEAT = 1
MY_PICKS = [1, 28, 29, 56, 57, 84, 85, 112, 113, 140, 141, 168, 169, 196, 197]

## Why a normal ADP board is the wrong sheet

Superflex does not change how anyone scores. It changes how many quarterbacks have a job — from 14
to 28 in this league — and that reprices the entire position. The same player, two markets:

In [2]:
q('''
    SELECT player_name, position,
           MAX(consensus_adp) FILTER (WHERE format = '1qb')       AS adp_1qb,
           MAX(consensus_adp) FILTER (WHERE format = 'superflex') AS adp_superflex
    FROM adp_consensus
    WHERE season = 2026 AND position = 'QB'
    GROUP BY 1, 2
    HAVING MAX(consensus_adp) FILTER (WHERE format = 'superflex') IS NOT NULL
    ORDER BY adp_superflex
    LIMIT 10
''')

,player_name,position,adp_1qb,adp_superflex
0,Josh Allen,QB,24.95,1.7
1,Drake Maye,QB,50.85,6.6
2,Lamar Jackson,QB,46.40,7.9
3,Joe Burrow,QB,54.05,8.8
4,Dak Prescott,QB,73.25,13.0
5,Jayden Daniels,QB,65.40,13.4
6,Jalen Hurts,QB,70.00,16.6
7,Matthew Stafford,QB,93.95,17.1
8,Brock Purdy,QB,95.90,18.5
9,Trevor Lawrence,QB,88.65,18.8


Josh Allen is the 1.01 of one market and an afterthought in the other. Any model reading the 1QB
board for this league is not slightly wrong — it is pricing a different game.

## Replacement level is why

A player is worth his projection **minus what the position costs for free**: the last player who
still has to start somewhere once all 14 teams fill their slots. Count the jobs and the whole board
falls out of it.

In [3]:
q('''
    SELECT DISTINCT position, starters_at_position AS jobs,
           round(replacement_level_points, 1) AS replacement_level
    FROM draft_board
    WHERE league_key = ? AND position IN ('QB','RB','WR','TE')
    ORDER BY position
''', [LEAGUE])

,position,jobs,replacement_level
0,QB,28,218.0
1,RB,31,136.7
2,TE,14,115.7
3,WR,39,139.1


28 quarterback jobs against roughly 32 startable NFL quarterbacks. That is the whole story: the
replacement quarterback is a backup, so an elite one is worth far more here than in a 1QB league,
where replacement is QB15 and perfectly startable.

## The board

Ranked on durability-adjusted points over replacement. `ol_tier` flags running backs behind a weak
offensive line — see below for why it is a warning rather than a ranking input.

In [4]:
q('''
    SELECT position AS pos, player_name, team, consensus_adp AS adp,
           round(points_over_replacement, 1) AS por,
           round(projected_points_adjusted, 1) AS proj,
           round(availability, 3) AS avail,
           round(projected_floor, 0) AS floor, round(projected_ceiling, 0) AS ceil,
           ol_tier
    FROM draft_board
    WHERE league_key = ? AND consensus_adp IS NOT NULL
    ORDER BY points_over_replacement DESC
    LIMIT 40
''', [LEAGUE])

,pos,player_name,team,adp,por,proj,avail,floor,ceil,ol_tier
0,RB,Bijan Robinson,ATL,2.6,178.4,315.1,0.970,311.0,341.0,Q4 best
1,RB,Jahmyr Gibbs,DET,1.8,178.1,314.7,0.950,322.0,343.0,Q3
2,QB,Josh Allen,BUF,1.7,160.5,378.5,0.985,368.0,391.0,None
3,WR,Puka Nacua,LA,4.9,129.7,268.8,0.920,275.0,298.0,None
4,RB,Jonathan Taylor,IND,5.7,129.2,265.9,0.920,260.0,303.0,Q4 best
5,WR,Ja'Marr Chase,CIN,7.9,129.2,268.3,0.972,270.0,278.0,None
6,WR,Amon-Ra St. Brown,DET,14.6,119.4,258.5,0.981,258.0,267.0,None
7,RB,De'Von Achane,MIA,21.8,118.8,255.4,0.962,259.0,277.0,Q2
8,WR,Jaxon Smith-Njigba,SEA,9.8,116.6,255.8,0.966,245.0,284.0,None
9,RB,James Cook,BUF,16.6,113.4,250.1,0.970,233.0,262.0,Q4 best


## The 1.01: Bijan, Gibbs, or Allen?

The board prices players one at a time, which cannot settle this — the question is what each choice
does to the *whole roster*. So each candidate is forced at the top pick and the draft is played out,
every candidate through the identical sampled rooms so the comparison is paired.

In [5]:
from scipy import stats
from src.gold.draft_plan import simulate_first_pick

first_pick = simulate_first_pick(LEAGUE, MY_SEAT, {
    "Josh Allen":     ["QB", "QB", "RB", "RB", "TE"],
    "Jahmyr Gibbs":   ["RB", "QB", "QB", "RB", "TE"],
    "Bijan Robinson": ["RB", "QB", "QB", "RB", "TE"],
}, trials=600)

wide = first_pick.pivot(index="trial", columns="player_name", values="points_vs_field")
summary = pd.DataFrame({
    "points_vs_field": wide.mean().round(1),
    "vs_allen": (wide.sub(wide["Josh Allen"], axis=0)).mean().round(1),
    "beats_allen": (wide.gt(wide["Josh Allen"], axis=0)).mean().round(3),
}).sort_values("points_vs_field", ascending=False)
summary

,points_vs_field,vs_allen,beats_allen
player_name,,,
Bijan Robinson,131.5,9.2,0.717
Jahmyr Gibbs,130.6,8.2,0.708
Josh Allen,122.4,0.0,0.000


In [6]:
for candidate in ["Bijan Robinson", "Jahmyr Gibbs"]:
    _, p = stats.ttest_rel(wide[candidate], wide["Josh Allen"])
    print(f"{candidate:<16} vs Josh Allen: p = {p:.2g}")

Bijan Robinson   vs Josh Allen: p = 1.5e-19
Jahmyr Gibbs     vs Josh Allen: p = 8.2e-16


**Take the running back.** Bijan and Gibbs both beat Allen, and the result is statistically
unambiguous — but the margin is only 8-9 points across a season of roughly 1700, and Allen still
wins nearly three rooms in ten. Two effects the model deliberately does *not* capture both lean
Allen's way: in-season replacement asymmetry (waiver running backs appear all year, waiver
quarterbacks do not) and a durability measure built only from injured-reserve stints, which misses
one- and two-game absences. Read this as a real but slim edge, not a mandate.

Bijan over Gibbs is under a point. Their offensive lines separate them slightly — ATL is
top-quartile, DET third — but treat them as interchangeable.

## The plan is worth more than the player

Every opening composition, drafted 300 times from seat 1 against a resampled room:

In [7]:
plans = q('''
    SELECT plan, round(starter_points, 1) AS points, round(points_vs_field, 1) AS vs_field,
           round(finish_rank, 2) AS avg_finish, round(win_rate, 3) AS win_rate,
           round(top_third_rate, 3) AS top_third
    FROM draft_plans
    WHERE league_key = ? AND draft_slot = ?
    ORDER BY points_vs_field DESC
''', [LEAGUE, MY_SEAT])

pd.concat([plans.head(8), plans[plans.plan == "best available"], plans.tail(4)])

,plan,points,vs_field,avg_finish,win_rate,top_third
0,2QB2RB1TE,1703.8,123.5,2.43,0.453,0.877
1,2QB2RB1WR,1695.2,112.4,2.66,0.350,0.840
2,2QB3RB,1688.6,107.9,2.94,0.320,0.797
3,2QB1RB1WR1TE,1686.3,103.8,3.06,0.347,0.803
4,2QB1RB2TE,1675.3,93.5,3.32,0.247,0.770
5,1QB2RB1WR1TE,1675.6,92.7,3.72,0.263,0.677
6,1QB3RB1TE,1668.8,88.2,3.88,0.247,0.660
7,2QB1RB2WR,1664.9,80.5,3.93,0.167,0.670
13,best available,1639.2,56.5,5.14,0.173,0.513
33,1RB4WR,1500.3,-87.2,10.44,0.010,0.083


**Every one of the best openings takes two quarterbacks.** Drafting pure best-available finishes
13th of 37 plans — choosing a plan at all is worth about 67 points, where choosing the right player
at 1.01 is worth 8. The plan is eight times the decision.

Best single ordering in the sweep: **RB, QB, QB, RB, TE**.

## Who actually reaches each of your picks

ADP is a mean and its standard deviation is the spread of real draft positions, so "still there at
pick k" is a tail probability rather than a guess. These are marginal probabilities — one player at
a time — so they do not add up across a position.

In [8]:
for pick in [1, 28, 29, 56, 57]:
    print(f"\n=== pick {pick} ===")
    print(q('''
        SELECT position AS pos, player_name, consensus_adp AS adp,
               round(points_over_replacement, 1) AS por, round(p_available, 2) AS p
        FROM draft_availability
        WHERE league_key = ? AND draft_slot = ? AND overall_pick = ? AND p_available >= 0.15
        ORDER BY points_over_replacement DESC
        LIMIT 6
    ''', [LEAGUE, MY_SEAT, pick]).to_string(index=False))


=== pick 1 ===
pos     player_name  adp   por   p
 RB  Bijan Robinson  2.6 178.4 1.0
 RB    Jahmyr Gibbs  1.8 178.1 1.0
 QB      Josh Allen  1.7 160.5 1.0
 WR      Puka Nacua  4.9 129.7 1.0
 RB Jonathan Taylor  5.7 129.2 1.0
 WR   Ja'Marr Chase  7.9 129.2 1.0

=== pick 28 ===
pos    player_name  adp  por    p
 RB  Ashton Jeanty 28.5 93.7 0.56
 WR    CeeDee Lamb 25.7 92.4 0.37
 QB         Bo Nix 30.9 90.5 0.70
 RB    Chase Brown 31.7 89.2 0.74
 RB Saquon Barkley 25.5 89.2 0.35
 QB    Jaxson Dart 29.6 86.3 0.61

=== pick 29 ===
pos    player_name  adp  por    p
 RB  Ashton Jeanty 28.5 93.7 0.50
 WR    CeeDee Lamb 25.7 92.4 0.31
 QB         Bo Nix 30.9 90.5 0.65
 RB    Chase Brown 31.7 89.2 0.69
 RB Saquon Barkley 25.5 89.2 0.29
 QB    Jaxson Dart 29.6 86.3 0.56

=== pick 56 ===
pos   player_name  adp  por    p
 TE  Trey McBride 65.2 78.2 0.78
 QB  Tyler Shough 47.8 68.4 0.19
 TE  Brock Bowers 61.0 65.3 0.64
 RB D'Andre Swift 59.4 62.9 0.71
 RB  Bucky Irving 64.5 60.9 0.92
 QB   Jordan L

## The quarterback cliff

14 quarterbacks are gone by pick 28. This is what makes the second one urgent and why mock drafts
against autopick bots mislead — bots draft off general rankings and let quarterbacks fall a dozen
picks past their real superflex price.

In [9]:
q('''
    SELECT count(*) FILTER (WHERE consensus_adp <= 28) AS gone_by_28,
           count(*) FILTER (WHERE consensus_adp <= 57) AS gone_by_57,
           count(*) FILTER (WHERE projected_points_adjusted >= 250) AS starter_quality,
           count(*) AS priced
    FROM draft_board WHERE league_key = ? AND position = 'QB'
''', [LEAGUE])

,gone_by_28,gone_by_57,starter_quality,priced
0,14,20,22,120


## Running backs behind a bad offensive line

Backs drafted in the ADP top 100 behind a bottom-quartile line returned **-6.6** points of surplus
against their draft price and beat that price only 35% of the time. Behind a top-quartile line:
**+20.3** and 56%. A 27-point swing at p=0.0055, positive in 7 of 7 seasons — and measured against
ADP, so the market is not already charging for it.

It applies to running backs and nobody else. Against the same grade, quarterbacks score -0.147,
tight ends +0.113 and receivers -0.048 — all noise.

Left as a warning rather than folded into the ranking: the relationship is not monotonic (the third
quartile edges the fourth), so it identifies backs to be wary of rather than backs to chase.

In [10]:
q('''
    SELECT player_name, team, consensus_adp AS adp, round(ol_grade, 1) AS ol_grade,
           round(points_over_replacement, 1) AS por, round(availability, 3) AS avail
    FROM draft_board
    WHERE league_key = ? AND ol_tier = 'Q1 worst' AND consensus_adp <= 130
    ORDER BY consensus_adp
''', [LEAGUE])

,player_name,team,adp,ol_grade,por,avail
0,Ashton Jeanty,LV,28.5,21.0,93.7,0.950
1,Josh Jacobs,GB,34.3,29.0,80.8,0.978
2,Omarion Hampton,LAC,38.5,9.7,50.7,0.820
3,Kenneth Walker III,KC,39.7,16.1,85.3,0.955
4,Cam Skattebo,NYG,52.3,24.2,25.6,0.773
5,Quinshon Judkins,CLE,66.2,1.6,36.5,0.911
6,Bhayshul Tuten,JAX,67.0,35.5,33.6,0.950
7,Aaron Jones,MIN,122.4,24.2,-12.0,0.945
8,Jordan Mason,MIN,123.0,24.2,-11.3,0.889


## Late rounds: chase the shallow positions

The instinct to keep taking receivers is wrong here, and replacement level says why. This league
starts 39 receivers, so the pool is 39 deep and the 40th is free. It starts 14 tight ends and 14
defenses. Value lives where the pool is shallow.

In [11]:
for pick in [112, 140, 168]:
    print(f"\n=== best available by position at pick {pick} (P >= 0.5) ===")
    print(q('''
        SELECT position AS pos, player_name, consensus_adp AS adp,
               round(points_over_replacement, 1) AS por, round(p_available, 2) AS p
        FROM draft_availability
        WHERE league_key = ? AND draft_slot = ? AND overall_pick = ? AND p_available >= 0.5
        QUALIFY row_number() OVER (PARTITION BY position ORDER BY points_over_replacement DESC) = 1
        ORDER BY por DESC
    ''', [LEAGUE, MY_SEAT, pick]).to_string(index=False))


=== best available by position at pick 112 (P >= 0.5) ===


pos    player_name   adp  por    p
DST            HOU 118.1 40.9 0.74
 TE   Travis Kelce 137.9 33.6 0.94
  K Brandon Aubrey 127.7 20.8 0.89
 QB  Aaron Rodgers 112.9 20.7 0.54
 RB      RJ Harvey 116.0  0.0 0.60
 WR     Josh Downs 121.9 -0.7 0.76

=== best available by position at pick 140 (P >= 0.5) ===
pos       player_name   adp    por    p
DST               BAL 154.9   24.9 0.85
 TE     Jake Ferguson 139.8   10.2 0.51
  K    Harrison Mevis 143.9    5.7 0.69
 WR     Denzel Boston 158.0  -18.8 0.90
 RB      Alvin Kamara 150.6  -34.4 0.72
 QB Michael Penix Jr. 142.3 -152.6 0.56

=== best available by position at pick 168 (P >= 0.5) ===
pos     player_name   adp   por    p
DST              NO 168.8 -13.7 0.54
 WR Adonai Mitchell 167.8 -46.0 0.51
 RB     Tank Bigsby 167.9 -67.1 0.51


Late receivers price out *below replacement* — they are worth less than what waivers will hand you
for nothing.

## Kicker last. Always.

The board says the best kicker is worth about 21 points over the last startable one. That number is
only spendable if you can identify him in August, and you cannot:

In [12]:
repeatability = q('''
    WITH kickers AS (
        SELECT player_id, season, sum(fantasy_points) AS points
        FROM weekly_stats
        WHERE position = 'K' AND season_type = 'REG' AND season BETWEEN 2015 AND 2025
        GROUP BY 1, 2 HAVING count(DISTINCT week) >= 12
    ),
    skill AS (
        SELECT player_id, position, season, sum(fantasy_points_ppr) AS points
        FROM weekly_stats
        WHERE position IN ('QB','RB','WR','TE') AND season_type = 'REG'
          AND season BETWEEN 2015 AND 2025
        GROUP BY 1, 2, 3 HAVING count(DISTINCT week) >= 12
    ),
    pairs AS (
        SELECT 'K' AS position, a.points AS this_year, b.points AS next_year
        FROM kickers a JOIN kickers b ON b.player_id = a.player_id AND b.season = a.season + 1
        UNION ALL
        SELECT a.position, a.points, b.points
        FROM skill a JOIN skill b
          ON b.player_id = a.player_id AND b.position = a.position AND b.season = a.season + 1
    )
    SELECT position, count(*) AS season_pairs, round(corr(this_year, next_year), 3) AS repeatability
    FROM pairs GROUP BY 1 ORDER BY repeatability DESC
''')
repeatability

,position,season_pairs,repeatability
0,WR,687,0.737
1,TE,293,0.699
2,RB,399,0.635
3,QB,179,0.368
4,K,221,-0.011


**-0.011.** Kicker scoring is not repeatable at all, so the top of the position is unpickable and
the pick belongs at 197. Defenses are modestly repeatable (points allowed carries year to year at
about +0.32), which argues for taking one in the last two or three rounds rather than dead last —
but the projection sources disagree with each other by more than the entire gap between the best
and worst startable defense, so do not reach.

## Bye weeks

A superflex slot accepts RB/WR/TE, so a quarterback bye does not require a third quarterback — it
requires a startable body. Check collisions before the last few picks, not after.

In [13]:
q('''
    SELECT f.bye, count(*) AS players,
           string_agg(b.player_name || ' (' || b.position || ')', ', '
                      ORDER BY b.points_over_replacement DESC) AS who
    FROM draft_board b
    JOIN (SELECT name, position, any_value(bye) AS bye FROM ffc_adp
          WHERE scoring_format = '2qb' AND season = 2026 GROUP BY 1, 2) f
      ON f.name = b.player_name AND f.position = b.position
    WHERE b.league_key = ? AND b.consensus_adp <= 60
    GROUP BY f.bye ORDER BY players DESC
''', [LEAGUE])

,bye,players,who
0,11.0,12,"Bijan Robinson (RB), Puka Nacua (WR), Jaxon Sm..."
1,6.0,11,"Jahmyr Gibbs (RB), Ja'Marr Chase (WR), Amon-Ra..."
2,8.0,8,"Christian McCaffrey (RB), Brock Purdy (QB), Ja..."
3,10.0,8,"Jalen Hurts (QB), Bo Nix (QB), Caleb Williams ..."
4,13.0,7,"Jonathan Taylor (RB), Lamar Jackson (QB), Derr..."
5,7.0,5,"Josh Allen (QB), Jayden Daniels (QB), Justin H..."
6,14.0,5,"CeeDee Lamb (WR), Jeremiyah Love (RB), Dak Pre..."
7,5.0,3,"Patrick Mahomes (QB), Tetairoa McMillan (WR), ..."


## The card

1. **1.01 — Bijan or Gibbs.** Allen is defensible and costs about 8 points.
2. **28 / 29 — two quarterbacks**, unless one of the top backs or receivers has fallen a long way.
   Realistic tier: Herbert, Dart, Nix. Not Hurts, not Dak — they will be gone.
3. **56 / 57 — running back and tight end.** Check availability above rather than a ranking.
4. **84 through 141 — shallow positions.** Tight end and defense outprice late receivers.
5. **196 / 197 — kicker last, always.**
6. Avoid the flagged backs unless the price has genuinely fallen to you.